# ClearBank - Análise Financeira com Python

Notebook do desafio final para leitura, validação e análise de transações bancárias a partir de um arquivo CSV, com geração de relatório mensal e exportação em JSON.

**Autor:** Thierry Vanden Broucke  
**Data:** 2026-05-18  
**Versão:** 1.0.3

## Estrutura do notebook

1. Imports e constantes
2. Funções utilitárias
3. Leitura e validação do CSV
4. Geração do relatório
5. Visualização com matplotlib
6. Exportação do JSON
7. Execução principal

In [1]:
import os
import csv
import json
os.environ["MPLCONFIGDIR"] = os.path.join(os.getcwd(), ".matplotlib")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from datetime import datetime

AUTOR = "Thierry Vanden Broucke"
DATA_VERSAO = "2026-05-18"
VERSAO = "1.0.3"

LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
ARQUIVO_GRAFICO = "grafico.png"

In [2]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def normalizar_texto(valor):
    return (valor or "").strip()


In [3]:
def validar_transacao(linha):
    try:
        id_texto = normalizar_texto(linha.get("id"))
        cliente_id = normalizar_texto(linha.get("cliente_id"))
        data_texto = normalizar_texto(linha.get("data"))
        tipo = normalizar_texto(linha.get("tipo")).lower()
        valor_texto = normalizar_texto(linha.get("valor"))
        descricao = normalizar_texto(linha.get("descricao"))
        categoria = normalizar_texto(linha.get("categoria"))

        if not id_texto or not id_texto.isdigit():
            return None

        if not cliente_id:
            return None

        if tipo not in ("credito", "debito"):
            return None

        try:
            valor = float(valor_texto)
        except ValueError:
            return None

        if valor <= 0:
            return None

        try:
            data_obj = datetime.strptime(data_texto, "%Y-%m-%d")
        except ValueError:
            return None

        return {
            "id": int(id_texto),
            "data": data_obj,
            "data_formatada": data_obj.strftime("%Y-%m-%d"),
            "mes": data_obj.strftime("%Y-%m"),
            "cliente_id": cliente_id,
            "tipo": tipo,
            "valor": valor,
            "descricao": descricao,
            "categoria": categoria,
            "suspeita": valor > LIMITE_SUSPEITO,
        }
    except KeyError:
        return None


In [4]:
def ler_transacoes(caminho_arquivo):
    transacoes_validas = []
    ids_vistos = set()
    total_lidas = 0
    total_invalidas = 0
    total_duplicadas = 0

    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                total_lidas += 1
                transacao = validar_transacao(linha)

                if transacao is None:
                    total_invalidas += 1
                    continue

                if transacao["id"] in ids_vistos:
                    total_invalidas += 1
                    total_duplicadas += 1
                    continue

                ids_vistos.add(transacao["id"])
                transacoes_validas.append(transacao)

    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho_arquivo}")
        return [], {
            "total_lidas": 0,
            "total_validas": 0,
            "total_invalidas": 0,
            "total_duplicadas": 0,
        }

    resumo_limpeza = {
        "total_lidas": total_lidas,
        "total_validas": len(transacoes_validas),
        "total_invalidas": total_invalidas,
        "total_duplicadas": total_duplicadas,
    }

    return transacoes_validas, resumo_limpeza

In [5]:
def gerar_relatorio(transacoes_validas, resumo_limpeza):
    resumo_mensal = {}
    suspeitas = []

    for transacao in transacoes_validas:
        mes = transacao["mes"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": transacao["valor"],
                "menor_valor": transacao["valor"],
            }

        resumo_mensal[mes]["quantidade"] += 1

        if transacao["tipo"] == "credito":
            resumo_mensal[mes]["total_credito"] += transacao["valor"]
        else:
            resumo_mensal[mes]["total_debito"] += transacao["valor"]

        resumo_mensal[mes]["maior_valor"] = max(resumo_mensal[mes]["maior_valor"], transacao["valor"])
        resumo_mensal[mes]["menor_valor"] = min(resumo_mensal[mes]["menor_valor"], transacao["valor"])

        if transacao["suspeita"]:
            suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data_formatada"],
                "valor": transacao["valor"],
            })

    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = (dados["total_credito"] + dados["total_debito"]) / dados["quantidade"]

    if transacoes_validas:
        datas = [transacao["data"] for transacao in transacoes_validas]
        data_mais_antiga = min(datas)
        data_mais_recente = max(datas)
        dias_entre_datas = (data_mais_recente - data_mais_antiga).days
        periodo = {
            "data_inicial": data_mais_antiga.strftime("%Y-%m-%d"),
            "data_final": data_mais_recente.strftime("%Y-%m-%d"),
            "dias_entre_datas": dias_entre_datas,
        }
    else:
        periodo = {
            "data_inicial": None,
            "data_final": None,
            "dias_entre_datas": 0,
        }

    return {
        "autor": AUTOR,
        "data_versao": DATA_VERSAO,
        "versao": VERSAO,
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": resumo_limpeza["total_validas"],
        "total_transacoes_invalidas": resumo_limpeza["total_invalidas"],
        "total_transacoes_lidas": resumo_limpeza["total_lidas"],
        "total_transacoes_duplicadas": resumo_limpeza["total_duplicadas"],
        "periodo_analisado": periodo,
        "resumo_mensal": dict(sorted(resumo_mensal.items())),
        "transacoes_suspeitas": suspeitas,
    }

In [6]:
def exibir_relatorio(relatorio):
    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {relatorio['total_transacoes_lidas']}")
    print(f"Linhas válidas: {relatorio['total_transacoes_validas']}")
    print(f"Linhas inválidas: {relatorio['total_transacoes_invalidas']}")
    print(f"IDs duplicados descartados: {relatorio['total_transacoes_duplicadas']}")
    print()

    periodo = relatorio["periodo_analisado"]
    print("===== PERÍODO ANALISADO =====")
    print(f"{periodo['data_inicial']} -> {periodo['data_final']}")
    print(f"Dias entre a transação mais antiga e a mais recente: {periodo['dias_entre_datas']}")
    print()

    print("===== RELATÓRIO MENSAL =====")
    for mes, dados in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print()
    print("===== TRANSAÇÕES SUSPEITAS =====")
    if relatorio["transacoes_suspeitas"]:
        for transacao in relatorio["transacoes_suspeitas"]:
            print(
                f"ID: {transacao['id']} | Cliente: {transacao['cliente_id']} | "
                f"Data: {transacao['data']} | Valor: {formatar_moeda(transacao['valor'])}"
            )
    else:
        print("Nenhuma transação suspeita encontrada.")


def gerar_grafico(relatorio, caminho_saida):
    meses = list(relatorio["resumo_mensal"].keys())
    total_credito = [relatorio["resumo_mensal"][mes]["total_credito"] for mes in meses]
    total_debito = [relatorio["resumo_mensal"][mes]["total_debito"] for mes in meses]
    saldo = [relatorio["resumo_mensal"][mes]["saldo"] for mes in meses]

    def eixo_moeda(valor, _):
        return formatar_moeda(valor)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), constrained_layout=True)
    fig.patch.set_facecolor("#f8fafc")

    cores_credito = "#1d4ed8"
    cores_debito = "#f97316"
    cor_saldo = "#0f766e"

    axes[0].bar(meses, total_credito, color=cores_credito, label="Crédito", width=0.65)
    axes[0].bar(meses, total_debito, bottom=total_credito, color=cores_debito, label="Débito", width=0.65)
    axes[0].set_title("Volume mensal de transações", fontsize=16, fontweight="bold")
    axes[0].set_ylabel("Valor acumulado")
    axes[0].legend(frameon=False, ncol=2)
    axes[0].yaxis.set_major_formatter(FuncFormatter(eixo_moeda))
    axes[0].tick_params(axis="x", rotation=0)

    linha = axes[1].plot(meses, saldo, color=cor_saldo, linewidth=3, marker="o", markersize=8)[0]
    axes[1].fill_between(meses, saldo, color=cor_saldo, alpha=0.15)
    axes[1].axhline(0, color="#334155", linewidth=1.2, linestyle="--", alpha=0.8)
    axes[1].set_title("Evolução do saldo mensal", fontsize=16, fontweight="bold")
    axes[1].set_ylabel("Saldo")
    axes[1].set_xlabel("Mês")
    axes[1].yaxis.set_major_formatter(FuncFormatter(eixo_moeda))
    axes[1].tick_params(axis="x", rotation=0)

    for eixo in axes:
        eixo.set_facecolor("#ffffff")
        eixo.spines["top"].set_visible(False)
        eixo.spines["right"].set_visible(False)
        eixo.grid(axis="y", linestyle="--", alpha=0.25)

    for x, y in zip(meses, saldo):
        axes[1].annotate(
            formatar_moeda(y),
            (x, y),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9,
            color=cor_saldo,
            fontweight="bold",
        )

    fig.suptitle("ClearBank - Resumo visual das transações", fontsize=18, fontweight="bold", color="#0f172a")
    fig.savefig(caminho_saida, dpi=200, bbox_inches="tight")
    plt.close(fig)


def salvar_json(relatorio, caminho_saida):
    with open(caminho_saida, mode="w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)

In [7]:
transacoes_teste, resumo_teste = ler_transacoes(ARQUIVO_CSV)
print(resumo_teste)
print(f"Primeira transação válida: {transacoes_teste[0]}")

{'total_lidas': 171, 'total_validas': 159, 'total_invalidas': 12, 'total_duplicadas': 0}
Primeira transação válida: {'id': 1, 'data': datetime.datetime(2026, 1, 3, 0, 0), 'data_formatada': '2026-01-03', 'mes': '2026-01', 'cliente_id': 'PF-NORTE-001', 'tipo': 'credito', 'valor': 2937.75, 'descricao': 'Recebimento de bonus trimestral', 'categoria': 'bonus', 'suspeita': False}


In [8]:
linha_valida = {
    "id": "999",
    "data": "2026-06-15",
    "cliente_id": "CLI999",
    "tipo": "credito",
    "valor": "1500.50",
    "descricao": "Teste válido",
    "categoria": "salario",
}

linha_invalida = {
    "id": "",
    "data": "15/06/2026",
    "cliente_id": "",
    "tipo": "pix",
    "valor": "abc",
    "descricao": "Teste inválido",
    "categoria": "transferencia",
}

print(validar_transacao(linha_valida))
print(validar_transacao(linha_invalida))

{'id': 999, 'data': datetime.datetime(2026, 6, 15, 0, 0), 'data_formatada': '2026-06-15', 'mes': '2026-06', 'cliente_id': 'CLI999', 'tipo': 'credito', 'valor': 1500.5, 'descricao': 'Teste válido', 'categoria': 'salario', 'suspeita': False}
None


In [9]:
relatorio_teste = gerar_relatorio(transacoes_teste, resumo_teste)
print(relatorio_teste["periodo_analisado"])
print(f"Meses calculados: {list(relatorio_teste['resumo_mensal'].keys())}")
print(f"Quantidade de suspeitas: {len(relatorio_teste['transacoes_suspeitas'])}")

{'data_inicial': '2026-01-03', 'data_final': '2026-11-20', 'dias_entre_datas': 321}
Meses calculados: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10', '2026-11']
Quantidade de suspeitas: 8


In [10]:
gerar_grafico(relatorio_teste, ARQUIVO_GRAFICO)
print(f"Gráfico salvo em: {ARQUIVO_GRAFICO}")

Gráfico salvo em: grafico.png


In [11]:
def main():
    transacoes_validas, resumo_limpeza = ler_transacoes(ARQUIVO_CSV)
    relatorio = gerar_relatorio(transacoes_validas, resumo_limpeza)
    exibir_relatorio(relatorio)
    salvar_json(relatorio, ARQUIVO_JSON)
    gerar_grafico(relatorio, ARQUIVO_GRAFICO)
    return relatorio


relatorio_final = main()

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 171
Linhas válidas: 159
Linhas inválidas: 12
IDs duplicados descartados: 0

===== PERÍODO ANALISADO =====
2026-01-03 -> 2026-11-20
Dias entre a transação mais antiga e a mais recente: 321

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações: 13
  Total crédito: R$ 19.072,75
  Total débito:  R$ 3.885,20
  Saldo:         R$ 15.187,55
  Média:         R$ 1.766,00
  Maior valor:   R$ 4.581,75
  Menor valor:   R$ 81,40
Mês: 2026-02
  Transações: 15
  Total crédito: R$ 22.711,00
  Total débito:  R$ 17.227,99
  Saldo:         R$ 5.483,01
  Média:         R$ 2.662,60
  Maior valor:   R$ 12.285,99
  Menor valor:   R$ 117,40
Mês: 2026-03
  Transações: 15
  Total crédito: R$ 25.396,50
  Total débito:  R$ 17.624,19
  Saldo:         R$ 7.772,31
  Média:         R$ 2.868,05
  Maior valor:   R$ 12.722,99
  Menor valor:   R$ 153,40
Mês: 2026-04
  Transações: 15
  Total crédito: R$ 22.351,00
  Total débito:  R$ 6.742,40
  Saldo:         R$ 15.608,